In [1]:
from lazypredict.Supervised import LazyRegressor
import os
import numpy as np
import pandas as pd
# sklearn imports (used in lazypredict)
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import BayesianRidge
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import ElasticNetCV
from sklearn.linear_model import GammaRegressor
from sklearn.linear_model import HuberRegressor
from sklearn.linear_model import Lars
from sklearn.linear_model import LarsCV
from sklearn.linear_model import Lasso
from sklearn.linear_model import LassoCV
from sklearn.linear_model import LassoLars
from sklearn.linear_model import LassoLarsCV
from sklearn.linear_model import LassoLarsIC
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import OrthogonalMatchingPursuit
from sklearn.linear_model import OrthogonalMatchingPursuitCV
from sklearn.linear_model import PassiveAggressiveRegressor 
from sklearn.linear_model import PoissonRegressor
from sklearn.linear_model import RANSACRegressor
from sklearn.linear_model import Ridge
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import SGDRegressor
from sklearn.linear_model import TweedieRegressor
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import LinearSVR
from sklearn.svm import NuSVR
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import ExtraTreeRegressor
# import catboost # for lazypredict, although not sklearn
# import xgboost # for lazypredict, although not sklearn
# import lightgbm # for lazypredict, although not sklearn
import sys
sys.path.append(os.path.dirname(os.getcwd()))
import functions

Data wrangling for the input data to models

In [2]:
df = pd.read_parquet('../data/features_table.parquet')

# add an order_number to each row in df, in order of the order_date
df['order_number'] = df['order_date'].rank(method='first').astype(int)

# add a delivery_date column to df, which is the order_date plus the actual_days
df['delivery_date'] = df['order_date'] + pd.to_timedelta(df['actual_days'], unit='D')

# Do one-hot encoding now, so that the train and test columns match
features_data = functions.one_hot_encode(df, ['origin_country', 'us_destination_state'])

In [3]:
# Choose the run_date for the train-test split
run_date = features_data['order_date'].quantile(0.9)
print(run_date)
# The window of dates will be one week
print(run_date + pd.Timedelta(days=7))

2017-12-26 00:00:00
2018-01-02 00:00:00


Random Forest Model and other Regressors
* Use lazypredict to evaluate many machine learning regressors
* Original formulation for one date, for testing during development

In [4]:
# Get data
X_train, y_train, X_test, y_test, feature_cols = functions.train_test_split(
    features_data, run_date, window_days=7)

In [5]:
regressor = LazyRegressor(verbose=0,
                          ignore_warnings=False,
                          custom_metric=mean_absolute_percentage_error,
                          regressors=[
                                        # catboost.CatBoostRegressor,
                                        # xgboost.XGBRegressor, 
                                        # lightgbm.LGBMRegressor,
                                        LinearRegression,
                                        NuSVR,
                                        SVR,
                                        PoissonRegressor,
                                        Lasso,
                                        # GammaRegressor,
                                        ElasticNet,
                                        LarsCV,
                                        HuberRegressor,
                                        TweedieRegressor,
                                        ElasticNetCV,
                                        BayesianRidge,
                                        LassoLarsCV,
                                        MLPRegressor,
                                        LassoCV,
                                        LassoLarsIC,
                                        RidgeCV,
                                        Ridge,
                                        OrthogonalMatchingPursuit,
                                        ExtraTreeRegressor,
                                        HistGradientBoostingRegressor,
                                        BaggingRegressor,
                                        # LGBMRegressor,
                                        OrthogonalMatchingPursuitCV,
                                        GradientBoostingRegressor,
                                        SGDRegressor,
                                        LinearSVR,
                                        RandomForestRegressor,
                                        KNeighborsRegressor,
                                        # XGBRegressor,
                                        Lars,
                                        ExtraTreeRegressor,
                                        DecisionTreeRegressor,
                                        PassiveAggressiveRegressor,
                                        AdaBoostRegressor,
                                        DummyRegressor,
                                        LassoLars,
                                        GaussianProcessRegressor,
                                        KernelRidge,
                                        RANSACRegressor,
                                        TransformedTargetRegressor,
])
models, predictions = regressor.fit(X_train, X_test, y_train, y_test)


/Users/alberta/Documents/git_token/deliveries/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/alberta/Documents/git_token/deliveries/.venv/lib/python3.13/site-packages/sklearn/utils/deprecation.py:71: FutureWarning: Class PassiveAggressiveRegressor is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDRegressor(loss='epsilon_insensitive', penalty=None, learning_rate='pa1', eta0 = 1.0)` instead.
  warnings.warn(msg, category=FutureWarning)


In [6]:
# help(regressor.fit) # Note the nothing will be returned to "predictions"

In [7]:
models.sort_values('RMSE').head(10)

,Adjusted R-Squared,R-Squared,RMSE,mean_absolute_percentage_error,Time Taken
Model,,,,,
AdaBoostRegressor,1.911724,0.138928,1.166014,0.251355,0.031089
RANSACRegressor,1.913351,0.137391,1.167054,0.269182,0.146207
GradientBoostingRegressor,1.977405,0.076895,1.207284,0.274727,0.128178
HistGradientBoostingRegressor,2.036560,0.021027,1.243281,0.304515,0.197219
ExtraTreeRegressor,2.045468,0.012614,1.248612,0.310691,0.021269
ExtraTreeRegressor,2.045468,0.012614,1.248612,0.310691,0.016966
RandomForestRegressor,2.053682,0.004856,1.253507,0.305520,0.212122
DecisionTreeRegressor,2.089586,-0.029053,1.274685,0.317270,0.021317
ElasticNetCV,2.090384,-0.029807,1.275151,0.255729,0.109856


In [8]:
models.sort_values('mean_absolute_percentage_error').head(10)

,Adjusted R-Squared,R-Squared,RMSE,mean_absolute_percentage_error,Time Taken
Model,,,,,
AdaBoostRegressor,1.911724,0.138928,1.166014,0.251355,0.031089
LassoCV,2.105399,-0.043988,1.283901,0.251650,0.080622
LarsCV,2.105313,-0.043907,1.283851,0.252007,0.062212
LassoLarsCV,2.105313,-0.043907,1.283851,0.252007,0.063402
ElasticNetCV,2.090384,-0.029807,1.275151,0.255729,0.109856
OrthogonalMatchingPursuitCV,2.162605,-0.098016,1.316704,0.260051,0.028933
RANSACRegressor,1.913351,0.137391,1.167054,0.269182,0.146207
GradientBoostingRegressor,1.977405,0.076895,1.207284,0.274727,0.128178
OrthogonalMatchingPursuit,2.384704,-0.307776,1.436979,0.282524,0.015857


AdaBoost does better on RMSE and WAPE than Random Forest
- But is the sample size significant?
- And is the difference / improvement significant?

In [9]:
results, feature_importances = functions.return_regressor_results(model = AdaBoostRegressor(),
                                                                  df = df,
                                                                  run_date = run_date,
                                                                  X_train=X_train,
                                                                  y_train=y_train,
                                                                  X_test=X_test,
                                                                  y_test=y_test)

Predicted rows: 19
 MAE: 0.932 
 RMSE: 1.177 
 R2: 0.123 
 MAPE: 0.282


In [10]:
# create a new column for results, with the name concatentated from the value of model and the string "predictions"
# this column will have the same values as the field pred
results[f"{str(results.loc[0, 'model'])}_predictions"] = results["pred"]
results

,origin_country,us_destination_state,order_date,order_number,actual,pred,abs_err,model,AdaBoostRegressor()_predictions
0,Vietnam,PR,2017-12-26,681,4,4.29,0.29,AdaBoostRegressor(),4.29
1,Vietnam,PR,2017-12-26,682,3,4.29,1.29,AdaBoostRegressor(),4.29
2,Vietnam,CA,2017-12-26,683,2,4.29,2.29,AdaBoostRegressor(),4.29
3,Vietnam,PR,2017-12-27,684,4,4.29,0.29,AdaBoostRegressor(),4.29
4,Vietnam,PR,2017-12-27,685,3,4.29,1.29,AdaBoostRegressor(),4.29
5,Vietnam,OH,2017-12-28,686,6,4.29,1.71,AdaBoostRegressor(),4.29
6,Vietnam,MD,2017-12-28,687,4,4.29,0.29,AdaBoostRegressor(),4.29
7,Vietnam,OH,2017-12-28,688,4,4.29,0.29,AdaBoostRegressor(),4.29
8,Vietnam,OH,2017-12-29,689,5,4.29,0.71,AdaBoostRegressor(),4.29
9,Vietnam,MO,2017-12-29,690,5,4.29,0.71,AdaBoostRegressor(),4.29


In [11]:
rf_results, rf_feature_importances = functions.return_regressor_results(
    model=RandomForestRegressor(),
    df=df,
    run_date=run_date,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test
)
rf_results[f"{str(rf_results.loc[0, 'model'])}_predictions"] = rf_results["pred"]
rf_results

Predicted rows: 19
 MAE: 1.053 
 RMSE: 1.259 
 R2: -0.004 
 MAPE: 0.309


,origin_country,us_destination_state,order_date,order_number,actual,pred,abs_err,model,RandomForestRegressor()_predictions
0,Vietnam,PR,2017-12-26,681,4,4.20,0.20,RandomForestRegressor(),4.20
1,Vietnam,PR,2017-12-26,682,3,4.20,1.20,RandomForestRegressor(),4.20
2,Vietnam,CA,2017-12-26,683,2,3.88,1.88,RandomForestRegressor(),3.88
3,Vietnam,PR,2017-12-27,684,4,4.20,0.20,RandomForestRegressor(),4.20
4,Vietnam,PR,2017-12-27,685,3,4.62,1.62,RandomForestRegressor(),4.62
5,Vietnam,OH,2017-12-28,686,6,4.69,1.31,RandomForestRegressor(),4.69
6,Vietnam,MD,2017-12-28,687,4,2.83,1.17,RandomForestRegressor(),2.83
7,Vietnam,OH,2017-12-28,688,4,3.53,0.47,RandomForestRegressor(),3.53
8,Vietnam,OH,2017-12-29,689,5,4.69,0.31,RandomForestRegressor(),4.69
9,Vietnam,MO,2017-12-29,690,5,6.00,1.00,RandomForestRegressor(),6.00


In [12]:
results.merge(rf_results,
              left_on=['order_number', 'origin_country', 'us_destination_state', 'order_date', 'actual'], 
              right_on=['order_number', 'origin_country', 'us_destination_state', 'order_date', 'actual'])\
    [['order_number', 'origin_country', 'us_destination_state', 'order_date', 'actual',
      'AdaBoostRegressor()_predictions', 'RandomForestRegressor()_predictions']]

,order_number,origin_country,us_destination_state,order_date,actual,AdaBoostRegressor()_predictions,RandomForestRegressor()_predictions
0,681,Vietnam,PR,2017-12-26,4,4.29,4.20
1,682,Vietnam,PR,2017-12-26,3,4.29,4.20
2,683,Vietnam,CA,2017-12-26,2,4.29,3.88
3,684,Vietnam,PR,2017-12-27,4,4.29,4.20
4,685,Vietnam,PR,2017-12-27,3,4.29,4.62
5,686,Vietnam,OH,2017-12-28,6,4.29,4.69
6,687,Vietnam,MD,2017-12-28,4,4.29,2.83
7,688,Vietnam,OH,2017-12-28,4,4.29,3.53
8,689,Vietnam,OH,2017-12-29,5,4.29,4.69
9,690,Vietnam,MO,2017-12-29,5,4.29,6.00


In [13]:
# The MAPE for the AdaBoostRegressor() was 0.250 while the MAPE for the RandomForestRegressor() was 0.311,
# but does that imply that the AdaBoostRegressor() is consistently better than the RandomForestRegressor()?
# i.e. was this difference from 0.250 to 0.311 on the MAPE significant?  With a sample size of 19, probably not.
# To test this, we can do a paired t-test on the absolute percentage errors of the two models.

In [14]:
# Statistical Significance Testing for Model Comparison
# ======================================================

import numpy as np
from scipy import stats

# Get predictions from both models
merged_results = results.merge(rf_results,
                                left_on=['order_number', 'origin_country', 'us_destination_state', 'order_date', 'actual'], 
                                right_on=['order_number', 'origin_country', 'us_destination_state', 'order_date', 'actual'],
                                suffixes=('_ada', '_rf'))

# Calculate absolute percentage errors for each prediction
merged_results['ape_ada'] = np.abs((merged_results['actual'] - merged_results['pred_ada']) / merged_results['actual'])
merged_results['ape_rf'] = np.abs((merged_results['actual'] - merged_results['pred_rf']) / merged_results['actual'])

print("=" * 70)
print("STATISTICAL SIGNIFICANCE TEST: AdaBoost vs RandomForest")
print("=" * 70)

# 1. MAPE Comparison
mape_ada = merged_results['ape_ada'].mean()
mape_rf = merged_results['ape_rf'].mean()
print(f"\n1. MAPE Values:")
print(f"   AdaBoost MAPE: {mape_ada:.4f}")
print(f"   RandomForest MAPE: {mape_rf:.4f}")
print(f"   Difference: {abs(mape_ada - mape_rf):.4f}")

# 2. Paired t-test: Tests if the mean difference in errors is significant
print(f"\n2. PAIRED T-TEST (comparing errors on same samples):")
error_diff = merged_results['ape_ada'] - merged_results['ape_rf']
t_stat, t_pvalue = stats.ttest_1samp(error_diff, 0)
print(f"   t-statistic: {t_stat:.4f}")
print(f"   p-value: {t_pvalue:.4f}")
if t_pvalue < 0.05:
    print(f"   ✓ SIGNIFICANT at α=0.05 (reject null hypothesis)")
    print(f"     → The difference in errors IS statistically significant")
else:
    print(f"   ✗ NOT SIGNIFICANT at α=0.05 (fail to reject null hypothesis)")
    print(f"     → The difference in errors is NOT statistically significant")

# 3. Wilcoxon Signed-Rank Test (non-parametric alternative)
print(f"\n3. WILCOXON SIGNED-RANK TEST (non-parametric):")
wilcoxon_stat, wilcoxon_pvalue = stats.wilcoxon(merged_results['ape_ada'], merged_results['ape_rf'])
print(f"   Test statistic: {wilcoxon_stat:.4f}")
print(f"   p-value: {wilcoxon_pvalue:.4f}")
if wilcoxon_pvalue < 0.05:
    print(f"   ✓ SIGNIFICANT at α=0.05")
else:
    print(f"   ✗ NOT SIGNIFICANT at α=0.05")

# 4. Effect Size (Cohen's d)
print(f"\n4. EFFECT SIZE (Cohen's d):")
pooled_std = np.sqrt((np.std(merged_results['ape_ada'], ddof=1)**2 + 
                      np.std(merged_results['ape_rf'], ddof=1)**2) / 2)
cohens_d = (mape_ada - mape_rf) / pooled_std
print(f"   Cohen's d: {cohens_d:.4f}")
if abs(cohens_d) < 0.2:
    print(f"   Interpretation: NEGLIGIBLE effect size")
elif abs(cohens_d) < 0.5:
    print(f"   Interpretation: SMALL effect size")
elif abs(cohens_d) < 0.8:
    print(f"   Interpretation: MEDIUM effect size")
else:
    print(f"   Interpretation: LARGE effect size")

# 5. Distribution comparison
print(f"\n5. ERROR DISTRIBUTION COMPARISON:")
print(f"   AdaBoost - Mean: {mape_ada:.4f}, Std: {merged_results['ape_ada'].std():.4f}, Median: {merged_results['ape_ada'].median():.4f}")
print(f"   RandomForest - Mean: {mape_rf:.4f}, Std: {merged_results['ape_rf'].std():.4f}, Median: {merged_results['ape_rf'].median():.4f}")

# 6. Win-Loss Analysis
ada_wins = (merged_results['ape_ada'] < merged_results['ape_rf']).sum()
rf_wins = (merged_results['ape_rf'] < merged_results['ape_ada']).sum()
ties = (merged_results['ape_ada'] == merged_results['ape_rf']).sum()
total = len(merged_results)

print(f"\n6. WIN-LOSS ANALYSIS (per sample):")
print(f"   AdaBoost better: {ada_wins} ({ada_wins/total*100:.1f}%)")
print(f"   RandomForest better: {rf_wins} ({rf_wins/total*100:.1f}%)")
print(f"   Ties: {ties} ({ties/total*100:.1f}%)")

# # Binomial test: Is one model significantly better more often?
# from scipy.stats import binom_test
# if ada_wins > rf_wins:
#     binom_pval = binom_test(ada_wins, ada_wins + rf_wins, 0.5, alternative='greater')
#     better_model = "AdaBoost"
# else:
#     binom_pval = binom_test(rf_wins, ada_wins + rf_wins, 0.5, alternative='greater')
#     better_model = "RandomForest"
# print(f"   Binomial test p-value: {binom_pval:.4f}")
# if binom_pval < 0.05:
#     print(f"   ✓ {better_model} wins significantly more often (p < 0.05)")
# else:
#     print(f"   ✗ No significant winner (p >= 0.05)")

# print("\n" + "=" * 70)
# print("CONCLUSION:")
# print("=" * 70)

STATISTICAL SIGNIFICANCE TEST: AdaBoost vs RandomForest

1. MAPE Values:
   AdaBoost MAPE: 0.2826
   RandomForest MAPE: 0.3092
   Difference: 0.0266

2. PAIRED T-TEST (comparing errors on same samples):
   t-statistic: -0.8129
   p-value: 0.4269
   ✗ NOT SIGNIFICANT at α=0.05 (fail to reject null hypothesis)
     → The difference in errors is NOT statistically significant

3. WILCOXON SIGNED-RANK TEST (non-parametric):
   Test statistic: 66.0000
   p-value: 0.3956
   ✗ NOT SIGNIFICANT at α=0.05

4. EFFECT SIZE (Cohen's d):
   Cohen's d: -0.0810
   Interpretation: NEGLIGIBLE effect size

5. ERROR DISTRIBUTION COMPARISON:
   AdaBoost - Mean: 0.2826, Std: 0.3334, Median: 0.1420
   RandomForest - Mean: 0.3092, Std: 0.3244, Median: 0.2183

6. WIN-LOSS ANALYSIS (per sample):
   AdaBoost better: 11 (57.9%)
   RandomForest better: 7 (36.8%)
   Ties: 1 (5.3%)


In [15]:
# # How much weight did the model put on each feature?
# feature_importances = pd.DataFrame({'feature': feature_cols, 'importance': model.feature_importances_}).sort_values('importance', ascending=False)

# # How many observations were in the training set for each feature?
# feature_counts = pd.DataFrame({'feature': feature_cols, 'count': X_train[feature_cols].sum()}).sort_values('count', ascending=False)
# 
# # Merge feature importance and counts to see if there's a relationship between them
# feature_analysis = pd.merge(feature_importances, feature_counts, on='feature')
# feature_analysis.sort_values('importance', ascending=False)

Evaluation across multiple time periods (folds) for the Random Forest Model
* https://docs.google.com/document/d/1EltPNSXuPh4FkSjoET2h-M8zrT9lXf7oChPe-i-87OQ/edit?tab=t.0


In [16]:
# # Choose the run_date for the train-test split
# run_date = features_data['order_date'].quantile(0.5)
# print(run_date)
# # The window of dates will be one week
# print(run_date + pd.Timedelta(days=7))

# # Create a list of run_dates to loop through
# run_dates = pd.date_range(start=run_date, end=features_data["order_date"].max() - pd.Timedelta(days=14), freq='7D')
# run_dates

In [17]:
# mape_records = []
# for r in run_dates:
#     # Get data
#     X_train, y_train, X_test, y_test, feature_cols = functions.train_test_split(features_data, 
#                                                                                 r,
#                                                                                 window_days=7)
#     # Only run the model if there are rows in the test set for the given run_date
#     if(len(X_test) > 0):
#         # Testing data dimension columns
#         rf_data = (
#             df.loc[
#                 (df["order_date"] >= r)
#                 & (df["order_date"] <= r + pd.Timedelta(days=7)),
#                 ["origin_country", "us_destination_state", "order_date", "order_number"],
#             ]
#             .reset_index(drop=True)
#             .copy()
#         )
#         # Run the model
#         model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
#         model.fit(X_train, y_train)
#         # Get the predicted values for the test set
#         y_pred = model.predict(X_test)
#         results, mape = functions.summarize_results(rf_data,
#                                                           r,
#                                                           y_test,
#                                                           y_pred, 
#                                                           model_name='Random Forest',
#                                                           print_stats=False)
#         mape_records.append({'run_date': r, 'mape': round(mape, 3)})

# rf_mape_df = pd.DataFrame(mape_records)
# rf_mape_df

In [18]:
# # remove outliers from rf_mape_df and average the rest
# q_high = rf_mape_df['mape'].quantile(0.99)
# rf_mape_df_filtered = rf_mape_df[(rf_mape_df['mape'] <= q_high)]
# rf_mape_avg = rf_mape_df_filtered['mape'].mean()
# rf_mape_avg

In [19]:
# display(rf_mape_df_filtered)

In [20]:
# However, the if the lead times are longer than, for example, 30 days, 
#   then the Delivery Date will be a few weeks ahead of the Order Date 
# Run Date (Now)
# Order Date (Next Week),
# Delivery Date (e.g. more than 30 days ahead) 
# This means that the model accuracy would need to be tracked by the order_number
#   And this model accuracy + tracking would not be complete until the delivery date has passed for all orders in the test set

# Next step: Improve the evaluation method to account for this scenario

## Understanding the Results

**Key Insight**: Just because AdaBoost has a lower MAPE (0.250 vs 0.311) doesn't mean it's *consistently* better. Here's what to look for:

1. **Paired t-test p-value < 0.05?** → The difference is statistically significant
2. **Wilcoxon test p-value < 0.05?** → Non-parametric confirmation of significance  
3. **Effect Size (Cohen's d)?** → How large is the difference in practical terms?
   - Small d (0.2-0.5): Difference exists but may not be practically meaningful
   - Medium d (0.5-0.8): Meaningful difference
   - Large d (>0.8): Very meaningful difference
4. **Win-Loss Analysis**: Which model wins more often per individual prediction?
5. **Sample Size**: Larger samples = more reliable conclusions

**Bottom Line**: 
- If p-value > 0.05: The difference could easily be due to **random variation**. Neither model is consistently better.
- If p-value < 0.05 AND effect size is meaningful: AdaBoost is likely genuinely better.
- If p-value < 0.05 BUT effect size is negligible: Statistical significance doesn't equal practical significance.